# 第 13 課 - 智能體記憶


## Setup

This notebook demonstrates how to build a travel booking agent with **persistent memory** using the **Microsoft Agent Framework** (MAF).

You will learn how different types of agent memory — working, short-term, and long-term — affect how an agent retains and uses information across conversations.

**Prerequisites:**
- A Microsoft Foundry project with a deployed chat model (e.g. `gpt-4.1-mini`).
- Logged in with the Azure CLI — run `az login` in your terminal.
- `AZURE_AI_PROJECT_ENDPOINT` — your Microsoft Foundry project endpoint.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — the name of your deployed model.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## 代理記憶體的類型

AI 代理可以利用不同種類的記憶體，每種記憶體有其不同的用途：

### 工作記憶體
對話線程本身 —— 在單一會話中交換的訊息。代理可以回顧同一線程中較早的訊息以維持連貫性。在 MAF 中，這是透過 **`agent.create_session()`** 建立，並回傳一個 `AgentSession`。

### 短期記憶體
在任務或會話持續期間存在，但不會永久儲存的資訊。例如，代理可能在多輪規劃對話中累積事實，並利用這些資料產生最終行程。

### 長期記憶體
跨會話持續存在的偏好和事實。回訪的使用者不必重複其飲食限制或旅遊風格。長期記憶體通常由外部儲存支持 —— 資料庫、檔案或向量索引 —— 並透過工具呈現給代理。


## 使用會話的工作記憶

記憶最簡單的形式是對話會話。當你將同一個會話（透過 `agent.create_session()` 建立）傳遞給連續的 `agent.run()` 呼叫時，代理會看到整個對話的歷史並能回憶早期的細節。

讓我們建立一個旅遊代理並示範工作記憶。


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

該代理能正確回憶起預算，因為兩則訊息共用相同的對話階段。這是 <strong>工作記憶</strong> — 它只存在於對話階段的存續期間。

### 新的對話線程會發生什麼事？

如果我們建立一個 <strong>新的</strong> 對話階段，代理將不記得之前的對話：


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## 長期記憶模式

要記住使用者偏好設定<strong>跨會話</strong>，我們需要一個存在於對話線程之外的持久存儲。代理人通過<strong>工具</strong>存取此存儲——工具是它可以調用來保存和檢索資訊的函數。

以下我們實作一個簡單的記憶體偏好存儲（在生產環境中你會用資料庫或向量索引來支援）並將其暴露為代理人可使用的工具。

### 架構
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### 情境一 — 首次使用者預訂週年旅行

Sarah 是首次拜訪。系統代理應透過工具記錄她的偏好，並利用這些偏好推薦旅館。


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### 情境 2 — Sarah 幾週後回來

Sarah 開始一個<strong>全新的對話串</strong>（模擬一個新的會話）。工作記憶是空的，但長期偏好存儲中仍有她的資訊。代理人應該檢索該資訊並用來個人化推薦。


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## 摘要

在本課程中，你學習了三種類型的代理記憶體以及如何使用 Microsoft Agent Framework 實作它們：

| 記憶體類型 | MAF 機制 | 生存週期 |
|---|---|---|
| <strong>工作記憶</strong> | `agent.create_session()` | 單次對話 |
| <strong>短期記憶</strong> | 線程內的累積上下文 | 單一任務 / 會話 |
| <strong>長期記憶</strong> | 透過 `@tool` 函數存取外部存儲 | 跨會話 |

### 主要重點
1. **`agent.create_session()`** 提供工作記憶 — 代理能看到一個會話內的完整對話歷史。
2. <strong>新會話會失去上下文</strong> — 沒有長期記憶，代理無法回憶過去的對話。
3. **`@tool` 函數** 彌補這個缺口 — 它們讓代理能將資訊儲存並從持久化存儲中取回。
4. <strong>個人化會隨時間提升</strong> — 儲存越多的偏好，代理的建議越準確。

### 實際應用
- <strong>客服服務</strong>：記住客戶歷史與偏好
- <strong>個人助理</strong>：維持日或週的上下文
- <strong>醫療保健</strong>：追蹤病患資訊與偏好
- <strong>電子商務</strong>：根據購買歷史做個人化購物

### 下一步
- 將記憶體中的 dict 換成資料庫或向量存儲（如 Azure AI Search）
- 為時間敏感資訊加入記憶過期機制
- 建立有共享記憶體的多代理系統
- 探索 Cognee 筆記本以用知識圖譜支援的記憶體


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
